In [1]:
from argparse import ArgumentParser
from datetime import datetime
from torch.utils.data import DataLoader
import logging
import os
from models.lstm_proj_diff import LSTM
from utils.dataset import TranslateDataset
from utils.earlystopper import EarlyStopper
import pandas as pd
from logging import getLogger
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import json
from utils.warmup import WarmupScheduler

In [2]:
df_train = pd.read_parquet("data/tokenized_train.parquet")
df_eval = pd.read_parquet("data/tokenized_eval.parquet")

In [8]:
df_train.head()

,en,pt,en_length,pt_length,en_max_len_word,pt_max_len_word,en_tokens,pt_tokens
0,"These people built, in stone, objects which th...","Quando esses objetos se foram, eles começaram ...",158,159,7,13,"[5, 5139, 1134, 4400, 443, 781, 4462, 443, 130...","[5, 1599, 1930, 9621, 766, 1887, 443, 1031, 67..."
1,I've spent my whole life trying to win his aff...,"Passei minha vida tentando ganhar o amor, a ap...",101,95,11,9,"[5, 536, 471, 789, 3721, 891, 2146, 1317, 1938...","[5, 51, 796, 1280, 1105, 1225, 2300, 3979, 416..."
2,I see you manage my family every day with grac...,Vejo você gerenciar minha família todos os dia...,124,126,11,12,"[5, 536, 1259, 778, 916, 1435, 891, 1771, 1540...","[5, 57, 5054, 849, 1640, 5301, 758, 1105, 1841..."
3,"Five years ago, my friends moved to london, bu...","Há cinco anos, meus amigos se mudaram para lon...",111,122,12,13,"[5, 41, 3142, 1372, 1851, 443, 891, 2411, 5045...","[5, 3211, 3216, 1165, 443, 1909, 2506, 766, 36..."
4,"Illarion shevardnadze, don't come near me, or ...","Illarion shevardnadze, não se aproxime de mim ...",107,110,13,13,"[5, 44, 79, 1821, 1993, 3985, 4675, 71, 81, 85...","[5, 44, 79, 1821, 1993, 3985, 4675, 71, 81, 85..."


In [9]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer: Tokenizer = Tokenizer.from_file('artifacts/tokenizer_10000.json')

In [10]:
df_train.en.values[0]

'These people built, in stone, objects which they seem to have seen in the sky, in flight, at some point, no doubt, landed on the surface of the earth as well.'

In [11]:
df_train.en_tokens.values[0]

array([   5, 5139, 1134, 4400,  443,  781, 4462,  443, 1302, 6585, 1247,
        939, 4087,  737,  897, 2627,  781,  739, 4424,  443,  781, 6561,
        443,  760, 1187, 2374,  443,  857, 5959,  443, 4515,  765,  769,
        739, 6411,  770,  739, 2607,  736, 1521,  446,    6])

In [14]:
tokenizer.decode(df_train.pt_tokens.values[0])

'Quando esses objetos se foram , eles começaram a reproduzí - los em pedra para que pudessem lembrar que estes eram os instrumentos em que os deuses vieram a eles .'